# 1. Set up environment

In [ ]:
import requests

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
from PIL import Image

import IPython.display

# 2. Investigate gene essentiality probability data

## 2.1. Load the data and look at it

In [ ]:
def download(url, output_file):
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(output_file, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

In [ ]:
download("https://plus.figshare.com/ndownloader/files/51064631", "CRISPRGeneDependency.csv")

In [ ]:
# Load the dataset into a DataFrame.
df = pd.read_csv("CRISPRGeneDependency.csv")

In [ ]:
# Index by screen ID.
df.set_index(df.columns[0], inplace=True)

In [ ]:
# Display dataset summary.
df.info()

In [ ]:
# Display some of the records.
df.head()

## 2.2. Handle missing values

In [ ]:
# Calculate the proportion of missing values
missing_by_row = df.isnull().mean(axis=1)
missing_by_column = df.isnull().mean(axis=0)

# Create data for plotting
missing_data = [missing_by_row, missing_by_column]
titles = ["Missing Values - Cancer Screens (Rows)", "Missing Values - Genes (Columns)"]

# Plot histograms
for data, title in zip(missing_data, titles):
    plt.figure(figsize=(10, 5))
    plt.hist(data, bins=100, alpha=0.7, color='blue')
    plt.title(title)
    plt.xlabel("Proportion of Missing Values")
    plt.ylabel("Frequency")
    plt.grid()
    plt.show()

In [ ]:
# Calculate the number of columns (genes) to drop
genes_to_drop = df.columns[df.isnull().any()]
num_dropped_genes = len(genes_to_drop)

# Drop columns with at least one missing value
df_no_missing = df.drop(columns=genes_to_drop)

# Print the number of dropped genes
print(f"Number of dropped genes: {num_dropped_genes}")

# Recalculate and plot the histogram for rows
missing_by_row_cleaned = df_no_missing.isnull().mean(axis=1)

plt.figure(figsize=(10, 5))
plt.hist(missing_by_row_cleaned, bins=30, alpha=0.7, color='green')
plt.title("Missing Values - Cancer Screens (Rows) After Dropping Genes")
plt.xlabel("Proportion of Missing Values")
plt.ylabel("Frequency")
plt.grid()
plt.show()

## 2.3. Remove common essential genes

In [ ]:
download("https://plus.figshare.com/ndownloader/files/51064916", "CRISPRInferredCommonEssentials.csv")

In [ ]:
# Load the list of column names from the CSV file (ignoring the first row)
column_names = pd.read_csv("CRISPRInferredCommonEssentials.csv", header=None).iloc[1:, 0].tolist()

# Remove columns from df_cleaned that match the names in the file
df_cleaned = df_no_missing.drop(columns=[col for col in df_no_missing.columns if col in column_names])

In [ ]:
df_cleaned

## 2.4. PCA plots

In [ ]:
# Function to perform PCA and plot results with density
def plot_pca_density(data, title):
    pca_result = PCA(n_components=2).fit_transform(data)
    x, y = pca_result[:, 0], pca_result[:, 1]
    # Calculate point density
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    
    # Sort points by density for better visualization
    idx = z.argsort()
    x, y, z = x[idx], y[idx], z[idx]
    
    plt.figure(figsize=(10, 10))
    scatter = plt.scatter(x, y, c=z, cmap="rainbow", s=10)
    # plt.colorbar(scatter, label="Density")
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid()
    plt.show()

# Standardize data for rows and columns
scaler = StandardScaler()
data_row = scaler.fit_transform(df_cleaned)
data_col = scaler.fit_transform(df_cleaned.T)

# Titles for plots
titles = ["PCA - Cancer Screens", "PCA - Genes"]

# Loop through data and titles to create density plots
for data, title in zip([data_row, data_col], titles):
    plot_pca_density(data, title)

## 2.5. Plot hierarchically sorted matrix

In [ ]:
def hierarchical_sort(df_cleaned):
    # Hierarchical clustering for rows
    linkage_rows = linkage(df_cleaned, method='ward')
    # Hierarchical clustering for columns
    linkage_cols = linkage(df_cleaned.T, method='ward')
    row_order = leaves_list(linkage_rows)
    col_order = leaves_list(linkage_cols)
    return df_cleaned.iloc[row_order, col_order]

In [ ]:
# Sort the dataframe according to hierarchical clustering
df_sorted = hierarchical_sort(df_cleaned)

In [ ]:
def plot_heatmap(df_sorted, filename, cmap="seismic"):
    # Assuming df_sorted is your pandas dataframe with values between 0 and 1
    data = df_sorted.values

    # Convert normalized data to an array in [0, 255] range
    scaled_data = (data * 255).astype(np.uint8)

    # Use a colormap
    cmap = plt.get_cmap(cmap)
    
    # Apply colormap
    colored_data = cmap(scaled_data)

    # Convert to an RGB image
    img = Image.fromarray((colored_data * 255).astype(np.uint8), mode='RGBA')

    # Save the image
    img.save(filename)
    
    # Display the image. Not on by default to trim down notebook size.
    if display:
        IPython.display.display(IPython.display.Image(filename=filename))

# Heatmap plotting is disabled to cut down on the file size for display.
# plot_heatmap(df_sorted, "heatmap_sorted.png")

## 2.6. Test different probability cut-offs

In [ ]:
# Proportion of cell lines killed if using given cut-off

fig, axes = plt.subplots(3, 2, figsize=(10, 10))
cutoffs = [0.5, 0.7, 0.9, 0.95, 0.98, 0.99]

for ax, cutoff in zip(axes.flat, cutoffs):
    proportions = (df_cleaned > cutoff).mean()
    ax.hist(proportions, bins=50, edgecolor='k', range=(0, 1))
    ax.set_xlabel(f'Proportion of values > {cutoff}')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Proportion of cell lines killed with cutoff>{cutoff}')
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Number of hits per screen

fig, axes = plt.subplots(3, 2, figsize=(10, 10))

for ax, cutoff in zip(axes.flat, cutoffs):
    hits = (df_cleaned > cutoff).sum(axis=1)
    ax.hist(hits, bins=50, edgecolor='k')
    ax.set_xlabel(f'Distribution of hits with cutoff>{cutoff}')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Number of hits with cutoff>{cutoff}')

plt.tight_layout()
plt.show()

## 2.7. Check values for known essential genes

In [ ]:
essential_genes = [
    "RPL5 (6125)",      # Ribosomal component, crucial for protein synthesis.
    "PCNA (5111)",      # DNA replication, supports polymerase activity.
    "CDC20 (991)",      # Activator of the anaphase-promoting complex, essential for mitosis.
    "AARS1 (16)",       # Alanyl-tRNA synthetase 1, translation.
    "POLR2G (5436)",    # RNA polymerase II subunit G.
    "PFDN2 (5202)",     # Prefoldin 2, protein folding.
    "PSMA1 (5682)",     # Proteasome.
]

plt.figure(figsize=(16, 8))
for column in essential_genes:
    plt.hist(df_no_missing[column], bins=100, range=(0, 1), histtype='step', label=column)

# Add legend and labels
plt.legend()
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Multiple Histograms')
plt.show()

## 2.8. Re-plot the heatmap with 0.95 cut-off

In [ ]:
df_cutoff = (df_sorted > 0.95)
# Heatmap plotting is disabled to cut down on the file size for display.
# plot_heatmap(df_cutoff, "histogram_cutoff.png", cmap="magma")